# Single-shot pdSTL reach–avoid planning

A Gaussian belief must remain inside the workspace, avoid one asymmetric obstacle, and reach the goal during its configured time window. The result compares the generic straight-line initialization with the hard-selected optimized plan.

## 1. Load the configurable problem

Edit `configs/scenarios/reach_avoid.yaml` to move or resize the workspace, goal, and any number of rectangular obstacles.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from inside the repository.')
sys.path.insert(0, str(ROOT / 'src'))

from planning.runners import run_reach_avoid, setup_problem
from utils import get_device, load_config
from visualization.animation import animate_reach_avoid
from visualization.planning import plot_reach_avoid

CONFIG_PATH = ROOT / 'configs/scenarios/reach_avoid.yaml'
OUTPUT_DIR = ROOT / 'outputs/experiments/reach_avoid'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg = load_config(CONFIG_PATH)
problem = setup_problem(cfg, device=get_device(), with_environment=True)
title = cfg['scenario'].get('name', 'Reach–Avoid')
visual = cfg.get('visualization', {})
print(f"Scenario: {title}")
print(f"Obstacles: {len(problem.env.by_role('obstacle'))}")
print(f"Horizon: {cfg['H']} steps ({cfg['H'] * cfg['dt']:.1f} s)")

## 2. One pdSTL task

$$\varphi = \mathbf{G}_{[1,H]}(\text{inside workspace} \land \text{outside obstacle}) \land \mathbf{F}_{[30,H]}(\text{inside goal}).$$

There are no waypoint predicates and no replanning windows.

In [ ]:
formula = problem.env.get_specification(cfg['H'], cfg['goal_interval'])
initial = problem.planner.evaluate_controls(problem.rollout, problem.init_guess, spec=formula)
print(formula)

## 3. Optimize once

In [ ]:
result = run_reach_avoid(str(CONFIG_PATH), show=False, save=False)
display(Markdown(
    f"**Hard pdSTL interval:** $[{result.hard_interval[0]:.4f}, {result.hard_interval[1]:.4f}]$  \n"
    f"**Required level:** $\\alpha={result.alpha:.2f}$ — achieved: `{result.threshold_met}`"
))

## 4. Initial and selected plans with optimization diagnostics

In [ ]:
fig, axes = plot_reach_avoid(
    result,
    problem.env,
    initial=initial,
    dt=cfg['dt'],
    u_max=problem.dyn.u_max,
    title=title,
    ellipse_every=visual.get('ellipse_every', 4),
    save_path=None,
    show=False,
)
display(fig)

## 5. Predicted belief animation

The animation reveals the same one-shot plan over its prediction horizon; it does not replan.

In [ ]:
gif_path = OUTPUT_DIR / 'reach_avoid.gif'
animate_reach_avoid(
    result,
    problem.env,
    dt=cfg['dt'],
    title=title,
    fps=visual.get('animation_fps', 6),
    filename=gif_path,
    show=False,
)
display(Image(filename=str(gif_path)))